In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

def simular_cinematica_2r(l1=2.0, l2=1.5, l3=1.5, x=2.0, y=1.0, phi_deg=0, joint_signal=1):
    # Configuração inicial da figura
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_aspect('equal')
    ax.set_title("Cinemática Inversa: Braço 2R", fontsize=14)

    phi = np.radians(phi_deg)

    # 1. DESACOPLAMENTO CINEMÁTICO
    # Calcula a posição do punho (x_w, y_w) subtraindo a contribuição do último elo
    xw = x - l3 * np.cos(phi)
    yw = y - l3 * np.sin(phi)
    
    # Desenho do espaço de trabalho (Workspace)
    r_max_punho = l1 + l2
    r_min_punho = abs(l1 - l2)
    
    # Círculo verde para o limite máximo
    workspace_max = plt.Circle((0, 0), r_max_punho, color='lightgreen', alpha=0.2)
    ax.add_patch(workspace_max)
    
    # Círculo branco para a área central inalcançável (se os elos forem de tamanhos diferentes)
    if r_min_punho > 0:
        workspace_min = plt.Circle((0, 0), r_min_punho, color='white', alpha=1.0)
        ax.add_patch(workspace_min)
        borda_min = plt.Circle((0, 0), r_min_punho, color='red', fill=False, linestyle=':', alpha=0.5)
        ax.add_patch(borda_min)

    # Cálculo da fórmula do slide: cos(theta2)
    cos_theta2 = (xw**2 + yw**2 - l1**2 - l2**2) / (2 * l1 * l2)
    
    # Origem do braço
    x0, y0 = 0, 0
    
    # Lógica de alcance baseada nos limites de -1 e 1
    if cos_theta2 > 1:
        # Ponto fora do workspace (Muito longe)
        status = f"INALCANÇÁVEL: Alvo muito longe!\ncos(θ2) = {cos_theta2:.3f} > 1"
        cor_braco = 'red'
        # Braço estica apontando na direção do alvo
        angulo_alvo = np.arctan2(yw, xw)
        x1, y1 = l1 * np.cos(angulo_alvo), l1 * np.sin(angulo_alvo)
        x2, y2 = x1 + l2 * np.cos(angulo_alvo), y1 + l2 * np.sin(angulo_alvo)  
        
    elif cos_theta2 < -1:
        # Ponto fora do workspace (Muito perto da base)
        status = f"INALCANÇÁVEL: Alvo muito perto!\ncos(θ2) = {cos_theta2:.3f} < -1"
        cor_braco = 'red'
        # Braço dobra completamente apontando na direção do alvo
        angulo_alvo = np.arctan2(yw, xw)
        x1, y1 = l1 * np.cos(angulo_alvo), l1 * np.sin(angulo_alvo)
        x2, y2 = x1 - l2 * np.cos(angulo_alvo), y1 - l2 * np.sin(angulo_alvo)

    else:
        # Ponto dentro do workspace (Alcançável)
        status = f"ALCANÇÁVEL\ncos(θ2) = {cos_theta2:.3f}"
        cor_braco = 'blue'
        
        # Escolhemos a solução "Cotovelo para baixo" (Elbow down)
        sin_theta2 = joint_signal * np.sqrt(1 - cos_theta2**2)
        theta2 = np.arctan2(sin_theta2, cos_theta2)
        
        # Cálculo de theta1 usando atan2 para estabilidade
        k1 = l1 + l2 * cos_theta2
        k2 = l2 * sin_theta2
        theta1 = np.arctan2(yw, xw) - np.arctan2(k2, k1)
        
        # Posições das juntas
        x1, y1 = l1 * np.cos(theta1), l1 * np.sin(theta1)
        x2, y2 = xw, yw # O efetuador chega exatamente no alvo

    # A ferramenta sempre sai da posição final da junta 2 (x2, y2) no ângulo phi
    x3, y3 = x2 + l3 * np.cos(phi), y2 + l3 * np.sin(phi)
    
    # Desenhando os elos
    ax.plot([x0, x1], [y0, y1], color='black', linewidth=5, label='Elo 1 ($l_1$)')
    ax.plot([x1, x2], [y1, y2], color=cor_braco, linewidth=5, label='Elo 2 ($l_2$)')
    ax.plot([x2, x3], [y2, y3], color='orange', linewidth=4, label='Ferramenta ($l_3$)')
    
    # Desenhando as juntas e o alvo
    ax.plot(x0, y0, 'ko', markersize=8) # Junta base
    ax.plot(x1, y1, 'ko', markersize=8) # Junta intermediária
    ax.plot(x2, y2, 'mo', markersize=8, label='Punho Alvo $(x_w, y_w)$') # Punho real/alvo
    
    ax.plot(x, y, 'r*', markersize=15, label='Alvo $(x, y)$') # Alvo
    ax.plot(x3, y3, 'co', markersize=6) # Efetuador final

    # Linha tracejada mostrando a orientação desejada (phi)
    ax.plot([x, x + 1.5*np.cos(phi)], [y, y + 1.5*np.sin(phi)], 'r--', alpha=0.5)

    # Exibindo o painel de status
    caixa_texto = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
    ax.text(0.05, 0.95, status, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', bbox=caixa_texto)

    ax.legend(loc='lower right')
    plt.show()

# Interface Interativa
interact(simular_cinematica_2r,
         l1=FloatSlider(min=1.0, max=4.0, step=0.1, value=2.0, description='Elo l1'),
         l2=FloatSlider(min=1.0, max=4.0, step=0.1, value=1.5, description='Elo l2'),
         l3=FloatSlider(min=1.0, max=4.0, step=0.1, value=1.5, description='Elo l3'),
         x=FloatSlider(min=-5.0, max=5.0, step=0.1, value=2.0, description='Alvo X'),
         y=FloatSlider(min=-5.0, max=5.0, step=0.1, value=2.0, description='Alvo Y'),
         phi_deg=FloatSlider(min=-180, max=180.0, step=1, value=0.0, description='Alvo phi'),
         joint_signal=IntSlider(min=-1, max=1, step=2, value=1, description='Cotovelo'),
        );

interactive(children=(FloatSlider(value=2.0, description='Elo l1', max=4.0, min=1.0), FloatSlider(value=1.5, d…